In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [8]:
from dataset.preprocess import PHASE_LABEL_COL, RECTIFIED_LEFT_IMAGE_PATH_KEY
from dataset.dataloader import EPISODE_FILENAME


def add_episode_path(episode_path, straightness_dataframe, delay = 4):
    episode = pd.read_csv(episode_path)
    phases = episode[PHASE_LABEL_COL].values  # Convert to numpy for np.where
    # Phase 2: all not straight
    phase_2_indices = np.where(phases == 2)[0]
    # Phase 3: split into groups, take last few as straight
    phase_3_indices = np.where(phases == 3)[0]
    if len(phase_3_indices) > 0:
        split_points = np.where(np.diff(phase_3_indices) != 1)[0] + 1
        groups = np.split(phase_3_indices, split_points)
        not_straight_from_3 = []
        straight_from_3 = [] 
        for group in groups:
            if len(group) > 0:
                num_straight = min(len(group), delay)
                straight_from_3.extend(group[-num_straight:])
                not_straight_from_3.extend(group[:-num_straight] if num_straight < len(group) else [])
    else:
        not_straight_from_3 = []
        straight_from_3 = []
    
    # Combine not straight: phase 2 + parts of phase 3
    not_straight_indices = sorted(list(phase_2_indices) + not_straight_from_3)
    not_straight_frame_paths = episode.loc[not_straight_indices, RECTIFIED_LEFT_IMAGE_PATH_KEY].tolist()
    
    # Phase 4: all straight
    phase_4_indices = np.where(phases == 4)[0]
    
    # Combine straight: parts of phase 3 + all phase 4
    straight_indices = sorted(straight_from_3 + list(phase_4_indices))
    straight_frame_paths = episode.loc[straight_indices, RECTIFIED_LEFT_IMAGE_PATH_KEY].tolist()
    
    new_rows = pd.DataFrame({
        "image": not_straight_frame_paths + straight_frame_paths,
        "label": [0] * len(not_straight_frame_paths) + [1] * len(straight_frame_paths)
    })
    straightness_dataframe = pd.concat([straightness_dataframe, new_rows], ignore_index=True)
    return straightness_dataframe
def create_straightness_dataframe(root_paths=(Path("/mnt/cluster/datasets/bowel_retraction/v1"), Path("/mnt/cluster/datasets/bowel_retraction/v2"))):
    straightness_dataframe = pd.DataFrame(columns=["image", "label"])
    for root_path in root_paths:
        episode_dirs = [d for d in root_path.iterdir() if d.is_dir() and (d / EPISODE_FILENAME).exists()]
        for episode_dir in episode_dirs:
            episode_path = episode_dir / EPISODE_FILENAME
            straightness_dataframe = add_episode_path(episode_path, straightness_dataframe)
    return straightness_dataframe
    
    


In [9]:
df = create_straightness_dataframe()


,image,label
0,/mnt/cluster/datasets/bowel_retraction/v1/2025...,0
1,/mnt/cluster/datasets/bowel_retraction/v1/2025...,0
2,/mnt/cluster/datasets/bowel_retraction/v1/2025...,0
3,/mnt/cluster/datasets/bowel_retraction/v1/2025...,0
4,/mnt/cluster/datasets/bowel_retraction/v1/2025...,0
...,...,...
8455,/mnt/cluster/datasets/bowel_retraction/v2/2025...,1
8456,/mnt/cluster/datasets/bowel_retraction/v2/2025...,1
8457,/mnt/cluster/datasets/bowel_retraction/v2/2025...,1
8458,/mnt/cluster/datasets/bowel_retraction/v2/2025...,1


In [11]:
df.to_csv("/mnt/cluster/datasets/bowel_retraction/evaluator/straightness_dataset.csv")